# Fast Fuzzy Filter
Using Fast Fuzzy Search to filter Potential Event-containing news. The main goal is to reduce the training time by achieving a more reduced but higher quality news set.

In [ ]:
from rapidfuzz import fuzz
import pandas as pd
import json

In [ ]:
# this cell is for loading the first version of fast fuzzy search filtered news, which we generated with the initial keyword set
df = pd.read_csv("filtered_news/filtered_news_with_keywords.csv",index_col=0)
df

,Date,Article_title,Stock_symbol,Url,Article,MatchedKeyword,MatchScore
1,2021-10-19 00:00:00 UTC,"""9 Foolish Truths That I Hold to Be Self-Evident""",NVDA,https://www.nasdaq.com/articles/9-foolish-trut...,"If you're a longtime listener to this podcast,...",covid,100.0
2,2020-03-25 00:00:00 UTC,"""Airlines"" Didn't Waste All Their Cash Flow on...",AAL,https://www.nasdaq.com/articles/airlines-didnt...,"Last Monday, Bloomberg reported that the top f...",covid,100.0
3,2018-09-22 00:00:00 UTC,"""Alexa, Make Me Money"": Conversational AI Prep...",ORCL,https://www.nasdaq.com/articles/alexa-make-me-...,We've already gotten comfortable with digital ...,new legislation,100.0
5,2022-08-27 00:00:00 UTC,"""Ape"" Into AMC? Buy These Explosive Growth Sto...",BILL,https://www.nasdaq.com/articles/ape-into-amc-b...,"Some investors ""aped"" into AMC Entertainment (...",wildfire,100.0
10,2023-08-04 00:00:00 UTC,"""Big Tech"" Earnings Review: ETFs in Focus",AAPL,https://www.nasdaq.com/articles/big-tech-earni...,"Since late July, five major tech companies fro...",iphone,100.0
...,...,...,...,...,...,...,...
662384,2022-03-04 00:00:00 UTC,“Pixelmon” a Dud… And 4 Other Surprises This Week,XOM,https://www.nasdaq.com/articles/pixelmon-a-dud...,"InvestorPlace - Stock Market News, Stock Advic...",invasion,100.0
662386,2021-09-15 00:00:00 UTC,“Yes in My Backyard”: How the Shift from China...,CC,https://www.nasdaq.com/articles/yes-in-my-back...,"InvestorPlace - Stock Market News, Stock Advic...",missile,100.0
662387,2021-08-24 00:00:00 UTC,‪11 Recovery Stocks That Could Get a Vaccine S...,GLPI,https://www.nasdaq.com/articles/11-recovery-st...,Recovery stocks were all the rage heading into...,covid,100.0
662389,2022-05-23 00:00:00 UTC,📈 Two Charts Show That Biotech Is in for a Sto...,MSFT,https://www.nasdaq.com/articles/two-charts-sho...,"InvestorPlace - Stock Market News, Stock Advic...",violent,100.0


## Designing the keyword list
- Be careful, do not to include many company and country names (they may result in poor filtering)

In [ ]:
with open("new_events.json") as events_json:
    events = json.load(events_json)

event_titles = [event["text"] for year in events for event in events[year]]
event_titles

In [ ]:
# v1 KEYWORD LIST, less strict
event_keywords = [
    # POLITICAL - leadership changes
    "general election", "referendum", "runoff", "abdication","Secretary",
    "coronation", "monarch", "assassin","muerder", "coup de etat", "junta","political", "royal", "the king","the queen",
    "House of Representatives","Chancellor", "Senate","revoke",

    # POLITICAL - wars and armed conflicts
    " war ", "invasion", "occupation", "bombing", "terrorism", "attack","Armed Forces",
    "civil war", "insurgency", "rebellion", "uprising","battle",
    "airstrike", "missile", "ceasefire", "treaty", "accord", "violent", "terrorist","rebels","military","Hamas","Al Qaeda","Gaza",

    # POLITICAL - economy
    "recession", "crisis", "currency change",
    "trade agreement", "customs union", "fraud", "embezzlement", "corruption", "scandal", "Euro enters circulation","East African Community",

    # POLITICAL - climate and environment policies
    "paris agreement", "kyoto protocol", "cop", "ipcc", "unfccc", "climate conference",

    # POLITICAL - legal milestones
    "constitution", "legal","legislation","law", "court","independence","reform","summit","trial",

    # POLITICAL - social movements
    "protest", "riot", "revolution", "social movement", "social tension", "violence","lgtb","rights","black lifes","unrest","women",

    # POLITICAL - diplomatic relations
    "Assembly", "United Nations", "nato", "g7", "g20", "bloc","diplomatic","peace agreement",

    # NON POLITICAL - disasters and crisis
    "earthquake", "tsunami", "hurricane", "flood", "eruption", "fire","polio", "environmental disaster","environmental"
    "pandemic", "epidemic", "covid", "sars", "coronavirus","ebola", "outbreak", "quarantine", "massacre",
    "crash", "derailment", "explosion", "storm", "Southern Leyte mudslide", "Cyclone", "shoot","SAG-AFTRA","Typhoon","crush",

    # NON POLITICAL - science & tech
    "spacex", "vaccine", "crispr", "chatgpt launch", "artificial intelligence","moon","asteroid","satellite","spacecraft",
    "iphone", "android", "discovery","ipod","invention","collision","scien","extinct","climate change","global warning","pollution","space exploration",

    # NON POLITICAL - cultural
    "olympics", "worldcup", "eurovision", "grammy", "tiktok",
    "viral", "church","World Series","pope","world record","championship",

    # NON POLITICAL - cultural (births & deaths)
    "death","dies","birth",

    # NON POLITICAL - corporate
    "company founded", "startup launched", "new venture","bankruptcy", "fraud", "Pandora Papers",
    
    # dangerous words (may apppear too many times)
    # "Facebook", 
    # "Twitter", "Spotify",
    # "Google",
    # "Schengen", "European Union", "World Trade Organization","acquired", "Eurasian Economic Union","trade area","trade organization","agreement",
    # "founded","releas","launch","collaps"
    # "NASA",
    # "Israel","Palestina",
    # "Jeff Bezos",
    # "Google",
    # "kill", "weapon", "Bitcoin","crypto",
]


# v2 KEYWORD LIST, more strict and refined after testing v1
event_keywords = [
    # POLITICAL - leadership changes
    "general elections", "referendum", "abdicate", "brexit","General Secretary","presidential election","inaugurated as President",
    "coronation", "monarch", "assassin","murder", "coup d'etat", "the king","Queen Margrethe","First Secretary",
    "House of Representatives","Chancellor", "Senate","Buckingham Palace","royal family","Prince George of Wales","Elizabeth II","Charles III",

    # POLITICAL - wars and armed conflicts
    "invasion","bombing", "terrorism","Armed Forces","Rose Revolution","Tulip Revolution","drug war","Tigray War","Taliban offensive",
    "civil war", "uprising","Virginia Tech shooting","Colorado shooting","Osama bin Laden","Muammar Gaddafi",
    "air strike", "missile", "ceasefire", "treaty", "terrorist","Hamas","Al Qaeda","Gaza","armed conflict","mass killings","School shooting",
    "Operation 1027",

    # POLITICAL - economy
    "trade agreement", "Euro enters circulation","Guayana Esequiba crisis","Ecuadorian political crisis","corruption scandal",

    # POLITICAL - climate and environment policies
    "paris agreement", "kyoto protocol", "ipcc", "unfccc", "climate change conference","COP26","COP28","summit","oil spill",
    "Intergovernmental Panel on Climate Change", "Sixth Assessment Report", "Environment Programme",

    # POLITICAL - legal milestones
    "constitution", "new legislation","legalization", "Supreme Court","pension reform","judicial reform","Independence of Montenegro",

    # POLITICAL - social movements
    "protest", "riots", "social movement", "social tension", "lgtb","civil rights","civil liberties","black lifes"," unrest ","George Floyd","Hong Kong protest",
    "Geneva Consensus Declaration on Promoting Women's Health and Strengthening the Family","International Organization for Migration","Human Rights",

    # POLITICAL - diplomatic relations
    "Assembly", "United Nations", " nato ","diplomatic","peace agreement",

    # NON POLITICAL - disasters and crisis
    "earthquake", "tsunami", "hurricane", "eruption", "wildfire","polio", "environmental disaster","Storm Uri","Storm Daniel",
    "epidemic","ebola", "quarantine", "lock down", "massacre","Rana Plaza collaps",
    "Southern Leyte mudslide", "Cyclone", "mass shooting","SAG-AFTRA","Typhoon","natural disaster","factory fire",
    "public health emergency","COVID-19 protest","covid protest","Tropical Storm","Monkeypox",

    # NON POLITICAL - science & tech
    "spacex", "malaria vaccine", "chatgpt", "deepmind","moon","asteroid","spacecraft","Royal Astronomical Society",
    "first iphone", "scientific discovery","first iPod","invention","collision","Scientific breakthrough","Scientific milestone",
    "climate change","global warming","space exploration","AlphaGo","Gotthard Base Tunnel","Event Horizon Telescope",
    "nuclear scientist","Protein folding","AlphaFold","Arecibo Telescope","Arecibo Observatory","Blue Origin","National Ignition Facility",

    # NON POLITICAL - cultural
    "olympics", "worldcup", "eurovision", "grammy","World Series","championship","Academy Award",

    # NON POLITICAL - cultural (births & deaths)

    # NON POLITICAL - corporate and others
    "startup launch", "Chapter 11 bankruptcy", "Pandora Papers","Panama Papers","International Consortium of Investigative Journalists","Elon Musk purchases Twitter",
    "Schengen", "European Union", "World Trade Organization", "Eurasian Economic Union","World Health Organization", 
    "North Atlantic Treaty Organization","East African Community","Treaty on the Prohibition of Nuclear Weapons","Istanbul Convention",
    "International Criminal Court","Bangsamoro Autonomous Region","Regional Comprehensive Economic Partnership","RCEP","Brereton Report",
    "NASA","Notre-Dame Cathedral","United States Space Force","Royal Australian Air Force","Human Rights Council","Commission on Narcotic Drugs",
    "Israel","Palestina","Yeonpyeong","Hugo Chavez","Nelson Mandela","funeral","Michael Brown","African Continental Free Trade Area","Champlain Towers","International Space Station",
    "killing all", "Bitcoin","company merger","company acquisition","plane crash","BRICS","Avatar","Grand Theft Auto V","ship Ever Given","National Independence","United Kingdom government crises",
]

## Run the fast fuzzy search algorithm

In [36]:
# load news dataframe
df = pd.read_csv("filtered_news/filtered_news_only_SP_no_duplicates_v8.csv")

In [18]:
df = df.sample(10000)

df["Article"]

480956    Wall Street analysts expect Oracle (ORCL) to p...
40693     [Editor’s note: “7 Tech Industry Dividend Stoc...
195875    Stocks opened higher Tuesday, as the Dow rebou...
93546     Atlas Resource Partners, L.P. ( ARP ) will beg...
535217    For Immediate Release\nChicago, IL - December ...
                                ...                        
415706    Among the underlying components of the Russell...
634322    What happened\nShares of Kratos Defense & Secu...
342139    Investors with an interest in REIT and Equity ...
179603    We have retained our Neutral recommendation on...
523153    InvestorPlace - Stock Market News, Stock Advic...
Name: Article, Length: 10000, dtype: object

In [51]:
# too slow!!!
# texts = df["Article"].tolist()

# threshold = 95  

# def filter_texts(texts, keywords, threshold=80):
#     results = []
#     for text in texts:
#         for kw in keywords:
#             if fuzz.partial_ratio(kw.lower(), text.lower()) >= threshold:
#                 results.append((text,kw.lower(),fuzz.partial_ratio(kw.lower(), text.lower())))
#                 break
#     return results

# filtered_texts = filter_texts(texts, event_keywords, threshold)

# # for text in filtered_texts:
# #     print()
# #     for i in range(len(text)):
# #         print(text[i])
# #     print(text)

# print(len(filtered_texts))

In [ ]:
# faster!
threshold = 95
keywords_lower = [kw.lower() for kw in event_keywords]

def match_keywords(text):
    text_lower = text.lower()
    for kw in keywords_lower:
        if kw in text_lower:
            return pd.Series([kw, 100])
        score = fuzz.partial_ratio(kw, text_lower)
        if score >= threshold:
            return pd.Series([kw, score])
    return pd.Series([None, None])

df[["MatchedKeyword", "MatchScore"]] = df["Article"].apply(match_keywords)

filtered_df = df[df["MatchedKeyword"].notna()]

print(len(filtered_df))

82373


In [39]:
print(len(event_keywords))
filtered_df.MatchedKeyword.value_counts().head(20)

202


MatchedKeyword
chatgpt                  8986
referendum               5695
bitcoin                  5352
rcep                     4442
hurricane                3642
european union           3167
invasion                 3095
brexit                   2904
senate                   2850
israel                   2617
assembly                 2457
summit                   2164
presidential election    2034
moon                     1751
missile                  1684
climate change           1592
the king                 1440
protest                  1351
legalization             1158
wildfire                 1146
Name: count, dtype: int64

In [35]:
keyword = "summit"
df[df["MatchedKeyword"] == keyword].Article.values

array(["Fintel reports that on October 30, 2023, Summit Insights Group upgraded their outlook for Western Digital (NASDAQ:WDC) from Hold to Buy .\nAnalyst Price Forecast Suggests 23.84% Upside\nAs of October 5, 2023, the average one-year price target for Western Digital is 48.26. The forecasts range from a low of 31.31 to a high of $63.00. The average price target represents an increase of 23.84% from its latest reported closing price of 38.97.\nSee our leaderboard of companies with the largest price target upside.\nThe projected annual revenue for Western Digital is 16,108MM, an increase of 42.15%. The projected annual non-GAAP EPS is 4.08.\nWhat is the Fund Sentiment?\nThere are 1130 funds or institutions reporting positions in Western Digital. This is a decrease of 21 owner(s) or 1.82% in the last quarter. Average portfolio weight of all funds dedicated to WDC is 0.18%, a decrease of 1.23%. Total shares owned by institutions decreased in the last three months by 4.99% to 295,395K sh

In [9]:
df.head(45)

,Date,Article_title,Stock_symbol,Url,Article,MatchedKeyword,MatchScore
1,2021-10-19 00:00:00 UTC,"""9 Foolish Truths That I Hold to Be Self-Evident""",NVDA,https://www.nasdaq.com/articles/9-foolish-trut...,"If you're a longtime listener to this podcast,...",covid,100.0
2,2020-03-25 00:00:00 UTC,"""Airlines"" Didn't Waste All Their Cash Flow on...",AAL,https://www.nasdaq.com/articles/airlines-didnt...,"Last Monday, Bloomberg reported that the top f...",covid,100.0
3,2018-09-22 00:00:00 UTC,"""Alexa, Make Me Money"": Conversational AI Prep...",ORCL,https://www.nasdaq.com/articles/alexa-make-me-...,We've already gotten comfortable with digital ...,new legislation,100.0
5,2022-08-27 00:00:00 UTC,"""Ape"" Into AMC? Buy These Explosive Growth Sto...",BILL,https://www.nasdaq.com/articles/ape-into-amc-b...,"Some investors ""aped"" into AMC Entertainment (...",wildfire,100.0
10,2023-08-04 00:00:00 UTC,"""Big Tech"" Earnings Review: ETFs in Focus",AAPL,https://www.nasdaq.com/articles/big-tech-earni...,"Since late July, five major tech companies fro...",iphone,100.0
11,2017-12-07 00:00:00 UTC,"""Bitcoin Stocks"" Get a New Member as Cboe Laun...",CME,https://www.nasdaq.com/articles/bitcoin-stocks...,"The price of bitcoin, the oldest and largest c...",extinct,100.0
12,2023-09-13 00:00:00 UTC,"""Blockbuster Status"": 3 Bios to Buy and Hold",ABBV,https://www.nasdaq.com/articles/blockbuster-st...,Profitability in the Biotech Industry is Hard ...,chatgpt,100.0
17,2022-03-24 00:00:00 UTC,"""Chip Rally,"" NATO Results, Drive Market Indexes",NVDA,https://www.nasdaq.com/articles/chip-rally-nat...,Markets kept their early morning gains and — a...,nato,100.0
21,2020-09-04 00:00:00 UTC,"""Dogs of the Dow"" Update: Buy These 5.9% Divid...",CVX,https://www.nasdaq.com/articles/dogs-of-the-do...,I dig dividend stocks that keep a low profile....,covid,100.0
23,2016-06-23 00:00:00 UTC,"""Finding Dory"" Had a Record-Breaking $135.1 Mi...",CMCSA,https://www.nasdaq.com/articles/finding-dory-h...,Walt Disney Co. 's(NYSE: DIS) Finding Dory rak...,civil war,100.0


In [ ]:
# save the result
filtered_df.to_csv("filtered_news/filtered_news_with_keywords_v2.csv")
filtered_df.head(45)

,Date,Article_title,Stock_symbol,Url,Article,MatchedKeyword,MatchScore
1,2021-10-19 00:00:00 UTC,"""9 Foolish Truths That I Hold to Be Self-Evident""",NVDA,https://www.nasdaq.com/articles/9-foolish-trut...,"If you're a longtime listener to this podcast,...",summit,100.0
3,2018-09-22 00:00:00 UTC,"""Alexa, Make Me Money"": Conversational AI Prep...",ORCL,https://www.nasdaq.com/articles/alexa-make-me-...,We've already gotten comfortable with digital ...,new legislation,100.0
5,2022-08-27 00:00:00 UTC,"""Ape"" Into AMC? Buy These Explosive Growth Sto...",BILL,https://www.nasdaq.com/articles/ape-into-amc-b...,"Some investors ""aped"" into AMC Entertainment (...",wildfire,100.0
6,2022-12-29 00:00:00 UTC,"""Avatar"" Sequel Crosses $1 Billion at the Box ...",DIS,https://www.nasdaq.com/articles/avatar-sequel-...,"In this video, I will talk about Walt Disney (...",avatar,100.0
7,2019-06-26 00:00:00 UTC,"""Avengers: Endgame"" Has Already Beaten Avatar'...",DIS,https://www.nasdaq.com/articles/avengers%3A-en...,"Almost since its record-setting debut, Avenger...",summit,100.0
8,2019-06-19 00:00:00 UTC,"""Avengers: Endgame"" Takes One Last Run at the ...",DIS,https://www.nasdaq.com/articles/avengers%3A-en...,It looks like Avengers: Endgame will snag that...,avatar,100.0
11,2017-12-07 00:00:00 UTC,"""Bitcoin Stocks"" Get a New Member as Cboe Laun...",CME,https://www.nasdaq.com/articles/bitcoin-stocks...,"The price of bitcoin, the oldest and largest c...",bitcoin,100.0
12,2023-09-13 00:00:00 UTC,"""Blockbuster Status"": 3 Bios to Buy and Hold",ABBV,https://www.nasdaq.com/articles/blockbuster-st...,Profitability in the Biotech Industry is Hard ...,chatgpt,100.0
13,2019-11-15 00:00:00 UTC,"""Borderlands 3"" and ""Outer Worlds"" Will Help T...",TTWO,https://www.nasdaq.com/articles/borderlands-3-...,When most people think of Take-Two Interactive...,grand theft auto v,100.0
17,2022-03-24 00:00:00 UTC,"""Chip Rally,"" NATO Results, Drive Market Indexes",NVDA,https://www.nasdaq.com/articles/chip-rally-nat...,Markets kept their early morning gains and — a...,nato,100.0


In [ ]:
df = pd.read_csv("filtered_news/filtered_news_with_keywords_v2.csv")

In [10]:
df["Article"]

0        If you're a longtime listener to this podcast,...
1        We've already gotten comfortable with digital ...
2        Some investors "aped" into AMC Entertainment (...
3        In this video, I will talk about Walt Disney (...
4        Almost since its record-setting debut, Avenger...
                               ...                        
82368    For many, competing in the Olympics is the pin...
82369    InvestorPlace - Stock Market News, Stock Advic...
82370    InvestorPlace - Stock Market News, Stock Advic...
82371    InvestorPlace - Stock Market News, Stock Advic...
82372    InvestorPlace - Stock Market News, Stock Advic...
Name: Article, Length: 82373, dtype: object